# Ensembling — Rent Prediction (Stacking)

**Goal:** beat the baseline by combining several models with **stacking**.

**Features:** same 2 as the baseline: `bedrooms` and `area_sqm`.

**Target:** `rent` (monthly, BHD)

**Performance measure:** MAE (Mean Absolute Error), the average number of BHD the predictions are off by. Lower is better.

Baseline to beat: pruned decision tree, public score **156.77**.

## Get the data

In [1]:
# Load the dataset
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

train_df = pd.read_csv('../Data/property_data.csv')
test_df = pd.read_csv('../Data/test.csv')

train_df.shape

(32090, 41)

In [2]:
train_df[['bedrooms', 'area_sqm', 'rent']].head()

,bedrooms,area_sqm,rent
0,studio,43.0,245.0
1,1,700 sqft / 65 sqm,300.0
2,5,"4,306 sqft / 400 sqm",1200.0
3,3,160.0,600.0
4,1,80.0,450.0


## Data Cleaning
Same rules as in `Baseline.ipynb`:
- **bedrooms**: `studio` → 0, `"3+ Maid"` → 3, `"7+"` → 7
- **area_sqm**: `"1,399 sqft / 130 sqm"` → 130

In [3]:
def convert_bedrooms_to_number(bedrooms_value):
    if pd.isna(bedrooms_value):
        return np.nan
    bedrooms_text = str(bedrooms_value).lower()
    if bedrooms_text.startswith('studio'):
        return 0
    number_part = bedrooms_text.split('+')[0]
    return int(number_part)


def convert_area_to_sqm(area_value):
    if pd.isna(area_value):
        return np.nan
    area_text = str(area_value)
    sqm_part = area_text.split('/')[-1]
    sqm_part = sqm_part.replace('sqm', '').replace(',', '').strip()
    return float(sqm_part)


train_df['bedrooms'] = train_df['bedrooms'].apply(convert_bedrooms_to_number)
train_df['area_sqm'] = train_df['area_sqm'].apply(convert_area_to_sqm)

# delete rows with missing bedrooms
train_df = train_df.dropna(subset=['bedrooms', 'area_sqm'])

train_df[['bedrooms', 'area_sqm', 'rent']].describe()

,bedrooms,area_sqm,rent
count,32078.000000,32078.000000,32078.000000
mean,2.263327,173.966800,691.806035
std,1.192649,147.594391,6612.115905
min,0.000000,1.000000,50.000000
25%,2.000000,98.000000,330.000000
50%,2.000000,130.000000,450.000000
75%,3.000000,182.000000,700.000000
max,10.000000,5000.000000,950000.000000


### New step: remove extreme rents
The max rent is **950,000 BHD** a month, which is impossible for a rental. These are probably sale prices that were scraped by mistake.

Only a few dozen rows are above 10,000 BHD, but they matter a lot:
- One 950,000 row in the validation set adds ~150 BHD to the MAE on its own. This is why the baseline scored 269 on our split but 156 on Kaggle.
- Models like Random Forest, KNN and Linear Regression predict *averages*, and these values pull the averages up for everyone.

In [4]:
print('Rows with rent > 10,000:', (train_df['rent'] > 10000).sum())

train_df = train_df[train_df['rent'] <= 10000]

train_df['rent'].describe()

Rows with rent > 10,000: 30


count    32048.000000
mean       580.437282
std        402.947741
min         50.000000
25%        330.000000
50%        450.000000
75%        700.000000
max       8400.000000
Name: rent, dtype: float64

## Select the features and the target

In [5]:
X = train_df[['bedrooms', 'area_sqm']]
y = train_df['rent']

X.head()

,bedrooms,area_sqm
0,0.0,43.0
1,1.0,65.0
2,5.0,400.0
3,3.0,160.0
4,1.0,80.0


**Split the data**

80% to train the model, 20% to test it on listings it has never seen (same split as the baseline).

In [6]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

((25638, 2), (6410, 2))

## Simplest score: predict the median for every listing

A real model must beat this.

In [7]:
from sklearn.metrics import mean_absolute_error

median_rent = y_train.median()
pred_median = np.full(len(y_test), median_rent)

mean_absolute_error(y_test, pred_median)

256.70483619344776

## Base models
Stacking works best when the base models are **different** from each other, so each one makes different mistakes:

| Model | How it predicts |
|---|---|
| Pruned decision tree | the baseline model: median rent of similar listings |
| Random forest | average of many trees (bagging) |
| Gradient boosting | trees built one after another, each fixing the last one's errors. `loss='absolute_error'` optimises MAE directly |
| KNN | average rent of the 25 most similar listings. Needs scaling because `area_sqm` is much bigger than `bedrooms` |

In [8]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

tree_model = DecisionTreeRegressor(max_depth=6, min_samples_leaf=20,
                                   criterion='absolute_error', random_state=42)

forest_model = RandomForestRegressor(n_estimators=200, min_samples_leaf=20,
                                     random_state=42, n_jobs=-1)

boosting_model = GradientBoostingRegressor(loss='absolute_error', n_estimators=300,
                                           random_state=42)

knn_model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=25))

base_models = [('tree', tree_model),
               ('forest', forest_model),
               ('boosting', boosting_model),
               ('knn', knn_model)]

In [9]:
# Score each base model on its own
results = []

for model_name, model in base_models:
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    results.append({'model': model_name, 'train_mae': train_mae, 'test_mae': test_mae})

pd.DataFrame(results).round(1)

,model,train_mae,test_mae
0,tree,160.6,162.7
1,forest,159.3,162.7
2,boosting,158.6,159.7
3,knn,159.7,163.0


## Stacking

1. Each base model makes predictions with 5-fold cross-validation, so they are always predicting rows they did not train on.
2. Those predictions become the input features of a **final model**, which learns how much to trust each base model.

### Final model = Linear Regression (like in class)

In [10]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

stacking_linear = StackingRegressor(estimators=base_models,
                                    final_estimator=LinearRegression(),
                                    n_jobs=-1)

stacking_linear.fit(X_train, y_train)

mean_absolute_error(y_test, stacking_linear.predict(X_test))

162.10796984748146

### Final model = Median (quantile) regression
`LinearRegression` minimises the *squared* error, but the competition scores the *absolute* error.

`QuantileRegressor(quantile=0.5)` is a linear regression that predicts the **median** instead of the mean, so it minimises the absolute error, the same thing MAE measures. `alpha=0` turns off regularization.

In [11]:
from sklearn.linear_model import QuantileRegressor

stacking_model = StackingRegressor(estimators=base_models,
                                   final_estimator=QuantileRegressor(quantile=0.5, alpha=0, solver='highs'),
                                   n_jobs=-1)

stacking_model.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('tree', ...), ('forest', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",QuantileRegressor(alpha=0)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[DecisionTreeR...ndom_state=42), RandomForestR...ndom_state=42), GradientBoost...ndom_state=42), Pipeline(step...ighbors=25))])]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['bedrooms','area_sqm']"
final_estimator_ final_estimator_: estimatorThe regressor fit on the output of `estimators_` and responsible forfinal predictions.,QuantileRegressor,QuantileRegressor(alpha=0)
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,2


In [12]:
#MAE on train
mean_absolute_error(y_train, stacking_model.predict(X_train))

156.64207814140397

In [13]:
#MAE on test
mean_absolute_error(y_test, stacking_model.predict(X_test))

158.59812226626087

How much does the final model trust each base model? (the weights it learned)

In [14]:
pd.Series(stacking_model.final_estimator_.coef_,
          index=[model_name for model_name, model in base_models]).round(3)

tree       -0.072
forest      0.293
boosting    0.550
knn         0.211
dtype: float64

## Results

In [15]:
results.append({'model': 'stacking (linear)',
                'train_mae': mean_absolute_error(y_train, stacking_linear.predict(X_train)),
                'test_mae': mean_absolute_error(y_test, stacking_linear.predict(X_test))})
results.append({'model': 'stacking (median)',
                'train_mae': mean_absolute_error(y_train, stacking_model.predict(X_train)),
                'test_mae': mean_absolute_error(y_test, stacking_model.predict(X_test))})

pd.DataFrame(results).sort_values('test_mae').round(1)

,model,train_mae,test_mae
5,stacking (median),156.6,158.6
2,boosting,158.6,159.7
4,stacking (linear),159.2,162.1
1,forest,159.3,162.7
0,tree,160.6,162.7
3,knn,159.7,163.0


- Removing the extreme rents brought the validation MAE close to the Kaggle public score (~160), so our split is now a good guide.
- Stacking with the median final model is the best, but only by a few BHD: with only 2 features every model sees the same information.
- **Next step for a bigger gain:** add more features (property type, location/area, furnished, water & electricity included, latitude/longitude) from the cleaning in `main.ipynb`.

## Submission
Predict rent for every listing in `test.csv` with the stacking model.
The submission file has only 2 columns: `Property_id` (from test.csv) and `rent` (the prediction).

In [16]:
# Clean test.csv with the same functions as train
test_df['bedrooms'] = test_df['bedrooms'].apply(convert_bedrooms_to_number)
test_df['area_sqm'] = test_df['area_sqm'].apply(convert_area_to_sqm)

test_df[['bedrooms', 'area_sqm']].isna().sum()

bedrooms    2
area_sqm    0
dtype: int64

In [17]:
# Test rows are never deleted: fill missing values with the train median
X_submission = test_df[['bedrooms', 'area_sqm']].fillna(X_train.median())

In [18]:
# Retrain the stacking model on all the labelled data, then predict
stacking_model.fit(X, y)

pred_submission = stacking_model.predict(X_submission)

In [19]:
submission_df = pd.DataFrame({
    'Property_id': test_df['Property_id'],
    'rent': pred_submission,
})

submission_df.to_csv('../Submissions/stacking_submission_v1.csv', index=False)

print(submission_df.shape)
submission_df.head()

(3566, 2)


,Property_id,rent
0,25c91c33-a523-42f2-8c10-9f340005471d,444.133258
1,6bcb7804-9cd7-4792-8cbe-abb4de65058a,391.391954
2,7b733e67-ec21-4427-95f8-4be3d9d18f6b,1041.935651
3,feefd51f-a7d6-45b0-a9eb-200d9f9821b4,372.240742
4,cca286dd-ddc6-41c2-9584-f8e17461220f,420.273353
